# 01-1 — ReAct Agent Demo

Demonstrates the **ReAct (Reason + Act)** agent loop with built-in tools and the **Console** for clean interactive output.

The agent:
1. Receives a user query
2. Reasons about which tools to call
3. Executes tools and observes results
4. Iterates until it can produce a final answer

**Prerequisites**: `OPENAI_API_KEY` set in the environment.

In [1]:
import os
from ravi.configs.settings import settings

CHAT_MODEL = settings.CHAT_MODEL
OPENAI_API_KEY = settings.OPENAI_API_KEY

print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))
print("CHAT_MODEL:", CHAT_MODEL)

OPENAI_API_KEY set: True
CHAT_MODEL: openai/gpt-5.4-mini


## Imports

`create_model_client()` builds the correct provider client from `CHAT_MODEL`, and `AgentCatalog` is imported from the public API instead of a private module.

In [2]:
from ravi.core.agent_catalog import AgentCatalog
from ravi.core.agents.react_agent import ReActAgent
from ravi.core.tools.builtin_tools import CalculatorTool, GetCurrentTimeTool
from ravi.integrations.llm.factory import create_model_client
from ravi.core.memory.unbounded_memory import UnboundedMemory
from ravi.core.context.implementations import UnboundedContext
from ravi.console import Console

## Build the agent

All resources — model, memory, context, tools — go into the catalog first.
`ReActAgent` then reads them from there; only config knobs (numbers, booleans) are passed directly.

In [ ]:
tools = [CalculatorTool(), GetCurrentTimeTool()]

api_keys = {
    "openai": os.environ.get("OPENAI_API_KEY", ""),
    "anthropic": os.environ.get("ANTHROPIC_API_KEY", ""),
    "google": os.environ.get("GOOGLE_API_KEY", os.environ.get("GEMINI_API_KEY", "")),
    "groq": os.environ.get("GROQ_API_KEY", os.environ.get("GROK_API_KEY", "")),
    "openrouter": os.environ.get("OPENROUTER_API_KEY", ""),
}

# Build the catalog — the single source of truth for all agent resources
catalog = AgentCatalog()
catalog.register_model("primary", create_model_client(CHAT_MODEL, api_keys=api_keys))
catalog.register_memory("default", UnboundedMemory())
catalog.register_context("default", UnboundedContext())
for tool in tools:
    catalog.register_tool(tool)

# Only config params remain in the constructor
agent = ReActAgent(
    name="DemoBot",
    description="A helpful assistant for demonstration.",
    catalog=catalog,
    max_iterations=5,
    verbose=True,
)

console = Console(agent)
print(f"Agent '{agent.name}' ready with {len(agent.tools)} tools.")
print(f"Configured chat model: {CHAT_MODEL}")
print(f"Resolved client: {agent.model_client}")
print(f"Memory: {agent.memory}")

Agent 'DemoBot' ready with 3 tools.
Configured chat model: openai/gpt-5.4-mini
Resolved client: <ravi.integrations.llm.openai.openai_client.OpenAIClient object at 0x74fd553b5510>
Memory: <UnboundedMemory(messages=0)>


: 

## Single-shot run (non-streaming)

`console.run()` pretty-prints tool calls and the final answer.

In [4]:
result = await console.run(
    'What is the square root of 256 multiplied by 14? Also what time is it?'
)

You → What is the square root of 256 multiplied by 14? Also what time is it?

╭──────────────────────────────────────────────────── DemoBot ────────────────────────────────────────────────────╮
│                                                                                                                 │
│  The square root of 256 multiplied by 14 is 224.0.                                                              │
│                                                                                                                 │
│  The current time in UTC is 18:46 on March 15, 2026.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

completed · 3 steps · 3 tool calls · 892 tokens · 6.3s

## Streaming run

`console.run_stream()` prints tokens as they arrive.

In [5]:
# Reset memory so we start fresh
agent.reset()
result = await console.run_stream('Compute 15 factorial and tell me the current time.')

You → Compute 15 factorial and tell me the current time.

✖ calculator

{
  "error": "invalid syntax (<string>, line 1)",
  "expression": "15!"
}

✔ get_current_time

{
  "datetime": "2026-03-15T18:47:11.805715",
  "timezone": "UTC",
  "timestamp": 1773580631.805715
}

...

,

202

6

.

3 tool calls · 4.7s

## Interactive mode

`console.interactive()` gives you a chat REPL. Type `/help` for commands, `exit` to quit.

In [ ]:
# Uncomment to start interactive chat (type 'exit' to quit)
# agent.reset()
# await console.interactive()